# Phase 1 — Generate evaluation images (Colab)

This notebook executes `experiments/exp3_generate_eval.py` on a Colab GPU. The notebook is intentionally thin — all real logic lives in the repo, so this file rarely needs to change.

**Before running, make sure to switch the runtime to a GPU (Runtime > Change runtime type > T4 GPU).**

Expected runtime on a free T4: roughly 1–2 hours for the full 3600 images. For a smoke test, use `--limit-pairs 2`.

## 1. Clone the repository

Set `REPO_URL` to your own fork before running.

In [ ]:
REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"

import os, subprocess
REPO_DIR = "/projects/challenges-in-attribute-control"
if os.path.exists(REPO_DIR):
    print(f'{REPO_DIR} already exists — pulling latest.')
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

## 2. Install dependencies

We use `uv` for speed (1-2 minutes vs 5+ with pip).

In [ ]:
!pip install -q uv
!uv pip install -e . --system
!pip install google-colab

## 3. Authenticate with Hugging Face (one-time)

SD 1.5 requires accepting the model's license on Hugging Face. If you haven't already, visit https://huggingface.co/runwayml/stable-diffusion-v1-5 and accept the terms.

Then paste your access token below.

In [ ]:
from huggingface_hub import login
login("") 

## 4. Quick sanity check (2 pairs, ~2 minutes)

Before launching the full 3600-image run, verify everything works end-to-end on a tiny subset.

In [ ]:
!python experiments/exp3_generate_eval.py \
    --config configs/exp3_default.yaml \
    --output-root /content/eval_smoke \
    --limit-pairs 2

In [ ]:
from PIL import Image
from pathlib import Path
smoke_imgs = list(Path('/content/eval_smoke').rglob('*.png'))[:1]
print(f'Found {len(smoke_imgs)} image(s) in smoke output')
if smoke_imgs:
    display(Image.open(smoke_imgs[0]))

## 5. Full run (120 pairs × 30 images)

This generates the actual evaluation dataset. The script is idempotent — if Colab disconnects, just re-run this cell and it resumes from where it stopped.

In [ ]:
!python experiments/exp3_generate_eval.py \
    --config configs/exp3_default.yaml

## 6. Package the results for download

Mount Drive (recommended) or zip the output for direct download.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DEST = '/content/drive/MyDrive/binding-research/eval_images'
shutil.copytree('/content/binding-research/data/eval_images', DRIVE_DEST, dirs_exist_ok=True)
shutil.copytree('/content/binding-research/results',          '/content/drive/MyDrive/binding-research/results', dirs_exist_ok=True)
print(f'Copied to {DRIVE_DEST}')

In [ ]:
import shutil
shutil.make_archive('/content/eval_images', 'zip', '/content/binding-research/data/eval_images')
from google.colab import files
files.download('/content/eval_images.zip')